# Bronze Layer - Data Ingestion

## Imports

In [20]:
import os
from datetime import datetime, timezone

import pandas as pd

# Variables

In [21]:
run_timestamp = datetime.now(timezone.utc)

ingestion_timestamp = run_timestamp.strftime("%Y%m%dT%H%M%SZ")

In [22]:
datasets = {
    "communications": {
        "path": "../../samples/communication-samples.csv",
        "format": "csv",
        "expected_columns": [
            "external_id",
            "carrier_company_id",
            "broker_comany_id",
            "direction",
            "channel",
            "status",
            "created_at",
            "updated_at",
            "from_contact_type",
            "to_contact_type",
            "thread_id"
        ]
    },
    "brokers": {
        "path": "../../samples/broker-samples.txt",
        "format": "txt",
        "expected_columns": [
            "_c0",
            "_c1",
        ],
    },
    "carriers": {
        "path": "../../samples/carrier-samples.json",
        "format": "json",
        "expected_columns": [
            "id",
            "name",
        ],
    }
}

# path = "s3://bucket/communications/*.csv" # Reading from S3 bucket and if many files representing the same table

## Functions

In [23]:
def validate_schema(df, expected_columns):

    incoming_columns = set(df.columns)
    expected_columns = set(expected_columns)

    missing_columns = expected_columns - incoming_columns
    new_columns = incoming_columns - expected_columns

    if missing_columns:
        raise ValueError(
            f"Schema rejected. Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if new_columns:
        print(
            f"Schema evolution detected. "
            f"New columns: {sorted(new_columns)}"
        )

        return df

    print("Schema validation successful.")

    return df

In [24]:
def read_csv(path, expected_columns):
    df = pd.read_csv(
        path,
        dtype=str
    )

    return validate_schema(df, expected_columns)

In [25]:
def read_json(path, expected_columns):

    df = pd.read_json(
        path,
        dtype=str
    )

    validate_schema(df, expected_columns)

    return df

In [26]:
def read_txt(path, expected_columns):
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["_c0", "_c1"],
        dtype=str
    )

    return validate_schema(df, expected_columns)

In [27]:
def add_metadata(df, source_path):

    df = df.copy()

    df["_source_file"] = os.path.abspath(source_path)
    df["_ingested_at"] = run_timestamp

    return df

## Main

In [28]:
readers = {
    "csv": read_csv,
    "json": read_json,
    "txt": read_txt
}

In [29]:
for dataset_name, config in datasets.items():

    path = config["path"]
    file_format = config["format"]
    expected_columns = config.get("expected_columns")

    reader = readers.get(file_format)

    if reader is None:
        raise ValueError(
            f"Unsupported format: {file_format}"
        )

    df = reader(
        path,
        expected_columns
    )

    df_with_metadata = add_metadata(df, path)

    output_path = (
        f"../../data/bronze/{dataset_name}/"
        # f"ingestion_timestamp={ingestion_timestamp}"
    )
    # output_path = f"s3://freighthero-data/bronze/{dataset_name}"

    os.makedirs(
        output_path,
        exist_ok=True
    )

    df_with_metadata.to_parquet(
        f"{output_path}/data.parquet",
        index=False
    )

    print(
        f"Bronze dataset '{dataset_name}' saved successfully."
    )

Schema validation successful.
Bronze dataset 'communications' saved successfully.
Schema validation successful.
Bronze dataset 'brokers' saved successfully.
Schema validation successful.
Bronze dataset 'carriers' saved successfully.
